# Spotify archive cleaning workflow

This notebook validates the cleaning workflow implemented using the archive dataset. The analysis is based on the approved plan: `genre_l1` is the primary grouping, `track_popularity` is the primary metric, and `streams_total` remains a secondary, modeled/estimated metric.

The raw archive file is kept unchanged. Outputs are read from the generated `outputs/` folder.

In [ ]:
import json
from pathlib import Path
import pandas as pd

root = Path('..')
summary = json.loads((root / 'outputs' / 'validation_summary.json').read_text(encoding='utf-8'))
print('rows_after_cleaning =', summary['rows_after_cleaning'])
print('exact_duplicate_rows =', summary['exact_duplicate_rows'])
print('repeated_track_id =', summary['repeated_track_id'])
print('repeated_spotify_id =', summary['repeated_spotify_id'])
print('track_popularity_range =', summary['track_popularity_range'])
print('source_unchanged =', summary['source_unchanged'])

## Expected difficult-case records

Before checking the actual cleaned sample, the following expectations are documented:

1. A row with a missing `album_name` should remain valid and not be imputed.
2. A non-charting record should keep `chart_peak_position` and `weeks_on_chart` missing.
3. A row with `has_anomaly = True` should remain in the dataset and remain visible.
4. A high-popularity row should remain within `90 <= track_popularity <= 100` and have non-missing `genre_l1` and `decade`.
5. A modeled/estimated stream record must retain `streams_total_estimated = True` and a non-missing `streams_total`.

In [ ]:
sample = pd.read_csv(root / 'outputs' / 'cleaned_sample.csv')
display(sample[['track_id', 'album_name', 'chart_peak_position', 'weeks_on_chart', 'has_anomaly', 'track_popularity', 'streams_total_estimated', 'streams_total']])

print('sample_count =', len(sample))
print('missing_album_names =', int(sample['album_name'].isna().sum()))
print('non_charting_rows =', int((sample['chart_peak_position'].isna() & sample['weeks_on_chart'].isna()).sum()))
print('anomaly_rows =', int(sample['has_anomaly'].fillna(False).sum()))
print('high_popularity_rows =', int(sample['track_popularity'].between(90, 100).sum()))
print('estimated_stream_rows =', int(((sample['streams_total_estimated'] == True) & sample['streams_total'].notna()).sum()))

## Grouped average validation

The grouped validation excludes rows with missing `genre_l1`, missing `decade`, or missing `track_popularity` so that the aggregate mean reflects valid observations only.

In [ ]:
grouped = pd.read_csv(root / 'outputs' / 'genre_decade_summary.csv')
grouped.head(10)